In [ ]:
import os,math,random
import numpy as np
import pandas as pd

EARTH_RADIUS_NM=3440.0


# ====================== geometry ======================

def haversine(a,b,c,d):
    # distance nikal raha hai bas
    a1,b1,c1,d1=map(np.radians,[a,b,c,d])

    dl=c1-a1
    dn=d1-b1

    x=np.sin(dl/2)**2 + np.cos(a1)*np.cos(c1)*(np.sin(dn/2)**2)

    # pehle sqrt use kiya tha but galat aa raha tha
    # return 2*EARTH_RADIUS_NM*np.sqrt(x)

    return 2*EARTH_RADIUS_NM*np.arcsin(np.sqrt(x))


def bearing_between(a,b,c,d):
    # heading calculate
    a,b,c,d=map(np.radians,[a,b,c,d])
    dl=d-b

    x=np.sin(dl)*np.cos(c)
    y=np.cos(a)*np.sin(c)-np.sin(a)*np.cos(c)*np.cos(dl)

    return (np.degrees(np.arctan2(x,y))+360)%360


def move_by_heading(lat,lon,h,s,t):
    # simple move logic
    dist=(s/3600.0)*t
    ang=dist/EARTH_RADIUS_NM

    lat1=math.radians(lat)
    lon1=math.radians(lon)
    h1=math.radians(h)

    nl=math.asin(math.sin(lat1)*math.cos(ang)+math.cos(lat1)*math.sin(ang)*math.cos(h1))

    nlo=lon1+math.atan2(math.sin(h1)*math.sin(ang)*math.cos(lat1),
                       math.cos(ang)-math.sin(lat1)*math.sin(nl))

    return math.degrees(nl),math.degrees(nlo)


# ====================== TFR ======================

class TFRZone:

    def __init__(self,lat,lon):
        self.lats=np.array(lat)
        self.lons=np.array(lon)

        self.centroid_lat=float(np.mean(lat))
        self.centroid_lon=float(np.mean(lon))

        ang=np.arctan2(self.lats-self.centroid_lat,self.lons-self.centroid_lon)
        order=np.argsort(ang)

        self.lats=self.lats[order]
        self.lons=self.lons[order]

    def contains(self,lat,lon):
        # basic ray casting
        inside=False
        j=len(self.lats)-1

        for i in range(len(self.lats)):
            yi,xi=self.lats[i],self.lons[i]
            yj,xj=self.lats[j],self.lons[j]

            if ((yi>lat)!=(yj>lat)) and (lon<(xj-xi)*(lat-yi)/((yj-yi)+1e-12)+xi):
                inside=not inside
            j=i

        return inside


# ====================== belief ======================

class BeliefState:

    SIGMA_MIN=0.05
    SIGMA_MAX=5.0

    def __init__(self):
        self.sigma_nm=self.SIGMA_MIN

    def update(self,obs,spd):
        # obs true matlab real data
        if obs:
            self.sigma_nm=max(self.SIGMA_MIN,self.sigma_nm*0.4)
        else:
            grow=(spd/3600.0)*30
            self.sigma_nm=min(self.SIGMA_MAX,self.sigma_nm+grow)

        # alternate try kiya tha constant increment but useless
        # self.sigma_nm+=0.1


# ====================== state ======================

def discretize_state(tfr_dist,rel,threat,sep,th_rel):

    # tfr bin
    if tfr_dist<0: t=0
    elif tfr_dist<5: t=1
    elif tfr_dist<15: t=2
    elif tfr_dist<30: t=3
    else: t=4

    # direction
    if -22.5<=rel<=22.5: d=0
    elif 22.5<rel<=157.5: d=1
    elif -157.5<=rel<-22.5: d=2
    else: d=3

    h=1 if threat else 0

    if sep<3: s=0
    elif sep<5: s=1
    elif sep<10: s=2
    else: s=3

    if th_rel is None or s==3:
        tb=0
    else:
        if -22.5<=th_rel<=22.5: tb=1
        elif 22.5<th_rel<=157.5: tb=2
        elif -157.5<=th_rel<-22.5: tb=3
        else: tb=4

    return (t,d,h,s,tb)


# ====================== reward ======================

def reward(state,action):

    t,d,h,s,_=state
    r=-1

    if t==0: r-=2000
    elif t==1 and h: r-=150
    elif h: r-=40

    if s==0: r-=1000
    elif s==1: r-=120

    if d==0: r+=5
    elif d==3: r-=3

    if action in ("TURN_LEFT_20","TURN_RIGHT_20"): r-=1
    elif action in ("TURN_LEFT_45","TURN_RIGHT_45"): r-=3
    elif action in ("CLIMB","DESCEND"): r-=2

    return r


# ====================== agent ======================

class Aircraft:

    def __init__(self,data):
        self.data=data
        self.i=0

        row=data.iloc[0]
        self.lat=row["lat"]
        self.lon=row["lon"]
        self.alt=row["Altitude"]
        self.spd=row["Speed"]

        self.belief=BeliefState()

        self.done=False

    def step(self):

        if self.done: return

        if self.i<len(self.data)-1:
            self.i+=1
            r=self.data.iloc[self.i]

            self.lat=r["lat"]
            self.lon=r["lon"]

            # interpolated ka check
            self.belief.update(not r.get("interpolated",False),self.spd)

        else:
            self.done=True


# ====================== main ======================

def run(data):

    agents=[Aircraft(df) for df in data.values()]

    steps=max(len(df) for df in data.values())

    for _ in range(steps):

        for a in agents:
            a.step()

    return agents


if __name__=="__main__":

    print("starting...")

    # fake load
    # actual me csv load hota hai but yahan skip

    data={}

    # alternate socha tha direct json load kare but chhoda
    # data=json.load(...)

    print("done")